<a id="top"></a>

# Read a trace and locate a failure

```yaml
title: "Read a trace and locate a failure"
keywords: tracing, trace, span, TraceEvent, state, agent graph, failure signature, tool error, wrong arguments, runaway loop, premature termination, debugging
estimated duration: 35
```

> **Lesson:** L12 — What an agent generates: state, logs, traces & extracts.
> **Roadmap:** [objectives.md](../../../../docs/origin/lesson_roadmaps/L12/objectives.md)
> **Where this fits:** the intro laid out the map — a run generates **state, logs, traces** (observability) and **extracts** (data). This notebook lives in the **traces** corner; because a trace is largely *state captured over time*, reading a trace here is also reading the **state** the graph built up.
> Runs offline — no API key needed (scripted `FakeModel`).

## Contents

- [1. Setup](#1-setup)
- [2. From `print()` to a structured record](#2-from-print-to-a-structured-record)
- [3. Read a trace, narrate the run](#3-read-a-trace-narrate-the-run)
- [4. Locate a failure from the trace alone](#4-locate-a-failure-from-the-trace-alone)
- [5. Takeaways](#5-takeaways)

## 1. Setup

One setup cell: import the frozen `common/` library and build **five traces** we'll read for the rest of the lecture. Each trace comes from running the same reference L10 graph (`run`) against a *scripted* `FakeModel` — the model's moves are decided in advance, so every trace is identical on every run, with **no API key and no network**.

We never read these scripts again after this cell. From here on we only read the **traces** they produced — that's the whole point: you debug an agent by reading the record it left, not by re-running it.

In [1]:
from fluffy_potato_curriculum.common.agent_graph import RunResult, run
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import TOOLS

# Each RunResult below carries a `.trace` (a list of TraceEvent spans). We build
# them once here, then spend the rest of the lecture *reading* them.

# good -- a clean multi-step run that terminates naturally.
good = run(
    FakeModel(
        [
            tool_reply(tool_call("c1", "calculator", {"expression": "17*23"})),
            tool_reply(tool_call("c2", "lookup", {"city": "Tokyo"})),
            text_reply("17 * 23 is 391, and Tokyo has about 37,000,000 people."),
        ]
    ),
    TOOLS,
    "What is 17 * 23, and what is the population of Tokyo?",
)

# tool_error -- a tool raises; the ToolNode turns it into an error ToolMessage
# (status="error", the span's `error` field set); model gives up gracefully.
tool_error = run(
    FakeModel(
        [
            tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
            text_reply("I couldn't fetch that URL -- it errored."),
        ]
    ),
    TOOLS,
    "Fetch https://crash and tell me what it says.",
)

# wrong_args -- the call SUCCEEDS, but the model looked up the wrong city.
wrong_args = run(
    FakeModel(
        [
            tool_reply(tool_call("c1", "lookup", {"city": "Paris"})),
            text_reply("Tokyo has about 11,000,000 people."),  # wrong: that's Paris
        ]
    ),
    TOOLS,
    "What is the population of Tokyo?",
)

# runaway -- the same failing call repeats; the recursion cap forces a halt.
# Distinct call ids (r0, r1, ...) matter: the graph's add_messages reducer keys on
# message id, so repeating one *identical* reply would be de-duplicated and the
# cycle would stop early instead of running away.
runaway = run(
    FakeModel([tool_reply(tool_call(f"r{i}", "lookup", {"city": "Atlantis"})) for i in range(8)]),
    TOOLS,
    "What is the population of Atlantis?",
    max_steps=4,
)

# premature -- the model answers with no tool call at all; the answer is a guess.
premature = run(
    FakeModel(
        [
            text_reply("It's probably around 5 million."),  # never looked it up
        ]
    ),
    TOOLS,
    "What is the population of Tokyo?",
)

traces: dict[str, RunResult] = {
    "good": good,
    "tool_error": tool_error,
    "wrong_args": wrong_args,
    "runaway": runaway,
    "premature": premature,
}
print("built traces:", ", ".join(traces))

built traces: good, tool_error, wrong_args, runaway, premature


[↑ Back to top](#top)

## 2. From `print()` to a structured record

L10's graph *streamed* one chunk per node visit. That chunk was real information — and it was gone the moment the run ended. You can't filter it, count it, diff it against another run, or hand it to next lesson's evaluator. It's text that scrolled past.

A **trace** is the same information kept as a **structured record**: an ordered list of typed `TraceEvent` spans hanging off `RunResult.trace`. Same graph, same control flow — L12 only *observes* it. The difference is that a record is a thing you can still query an hour later.

Below: the second span of the `good` trace, two ways. `.one_line()` is the print-style rendering — readable, ephemeral-feeling. But it's backed by a real object whose fields you can reach into and compute on.

In [2]:
# trace[0] is the chain span (the run summary); trace[1] is the first model call.
first_llm = good.trace[1]

# The print-style view -- what L10 gave you, but reproducible:
print(first_llm.one_line())

# ...and the same span as a structured record you can query field by field:
print("  run_type:", first_llm.run_type)
print("  name:    ", first_llm.name)
print("  outputs: ", first_llm.outputs)  # e.g. {'tool_calls': ['calculator']}

llm   agent                -> tool_calls=['calculator'] tokens=120
  run_type: llm
  name:     agent
  outputs:  {'tool_calls': ['calculator']}


Because it's a record, "filter the trace" is just a list comprehension — something a `print()` stream can never give you. Here: every tool span across the whole run, with its arguments.

In [3]:
tool_spans = [span for span in good.trace if span.run_type == "tool"]
for span in tool_spans:
    print(f"{span.name:12} inputs={span.inputs}")

calculator   inputs={'expression': '17*23'}
lookup       inputs={'city': 'Tokyo'}


[↑ Back to top](#top)

## 3. Read a trace, narrate the run

**Objective 1: read a trace and narrate what the agent did, span by span.** The skill is to walk the spans top to bottom and tell the story out loud. `.one_line()` is built for exactly this.

The span order is the run order. `trace[0]` is always the `chain` span — the run summary, up top. After it, spans alternate `llm` (a model call) and `tool` (a dispatch the model asked for).

In [4]:
def narrate(result: RunResult) -> None:
    """Print one line per span, in run order -- the trace, read top to bottom."""
    for i, span in enumerate(result.trace):
        print(f"[{i}] {span.one_line()}")


narrate(good)

[0] chain agent_graph.run      {'termination': 'natural', 'iterations': 3}
[1] llm   agent                -> tool_calls=['calculator'] tokens=120
[2] tool  calculator           {'expression': '17*23'} -> '391'
[3] llm   agent                -> tool_calls=['lookup'] tokens=120
[4] tool  lookup               {'city': 'Tokyo'} -> '37000000'
[5] llm   agent                -> tool_calls=[] tokens=120


Read it aloud: *"The run started. The model called `calculator(expression='17*23')` and got `391`. It called `lookup(city='Tokyo')` and got `37000000`. Then it answered in text and stopped."* That's the whole run, reconstructed from the record — we never re-ran anything.

Notice **where the truth lives**: the `inputs` on each tool span. `calculator` ran on `'17*23'` and `lookup` ran on `'Tokyo'` — those arguments are the model's actual choices. A call-count summary ("called two tools") would hide that; the arguments don't.

### 3.1 The trace *is* the agent's state, captured

Look again at that alternating `llm` / `tool` sequence — it isn't just a list of events, it's the agent's **state** written down. On every `llm` span the model's input is the *entire conversation so far*: the question, plus every tool call and result before it. That growing message history is the live **state** the graph feeds back to the model each turn (the `add_messages` reducer appends to it; L10 built it up in memory); the trace is that same state **serialized over time**, so you can read it after the run instead of watching it live.

So *"read the trace"* and *"read the state the graph built"* are the same act. **State** is the first artifact a run generates (see the intro's map); a trace is how you make it durable.

### 3.2 The `RunResult` summary is *derived from* the trace

`RunResult` (from L10) is the summary; the trace is the full record it was derived from. You should be able to point at where each summary field came from:

- `final_text` — the text of the last `llm` span (the one with no tool calls).
- `iterations` — the count of `llm` spans.
- `termination` — recorded on the `chain` span's `outputs`.

Let's recompute all three from the trace alone and check they match the `RunResult`.

In [5]:
# iterations == number of llm spans
llm_spans = [span for span in good.trace if span.run_type == "llm"]
iterations_from_trace = len(llm_spans)

# termination == the chain span's recorded outcome (chain span is always trace[0])
chain_span = good.trace[0]
termination_from_trace = chain_span.outputs["termination"]

print("iterations:  trace =", iterations_from_trace, "| RunResult =", good.iterations)
print("termination: trace =", termination_from_trace, "| RunResult =", good.termination)

assert iterations_from_trace == good.iterations
assert termination_from_trace == good.termination
print("\nThe summary holds no information the trace doesn't -- it's a view onto the record.")

iterations:  trace = 3 | RunResult = 3
termination: trace = natural | RunResult = natural

The summary holds no information the trace doesn't -- it's a view onto the record.


[↑ Back to top](#top)

## 4. Locate a failure from the trace alone

**Objective 2: given a failed or wrong run, point to where it went off the rails and name the failure's signature — by reading, not re-running.** Re-running a non-deterministic agent might hide the bug or produce a different one; the trace of the failing run *is* the reproduction.

There are four signatures worth learning by name. We have one trace for each. Walk them.

### 4.1 Tool error — a tool failed, look at the `error` field

**Signature:** a `tool` span whose `error` field is set (`one_line()` tags it `[is_error]`). The tool raised; the `ToolNode` converted the exception into an error `ToolMessage` (`status="error"`) instead of crashing, and it was recorded on the span's `error` field. Read the model's *next* span to see what it did about it — here it gave up gracefully and answered in text.

In [6]:
narrate(tool_error)

# Find the failing span by reading the error field -- no re-running needed.
failed = [span for span in tool_error.trace if span.error is not None]
print("\nfailing span:", failed[0].name, "->", failed[0].error)

[0] chain agent_graph.run      {'termination': 'natural', 'iterations': 2}
[1] llm   agent                -> tool_calls=['flaky_fetch'] tokens=120
[2] tool  flaky_fetch          {'url': 'https://crash'} -> "Error: RuntimeError('connection reset by peer')\n Please fix your mistakes."  [is_error]
[3] llm   agent                -> tool_calls=[] tokens=120

failing span: flaky_fetch -> Error: RuntimeError('connection reset by peer')
 Please fix your mistakes.


### 4.2 Wrong arguments — the call *succeeded*, but on the wrong input

**Signature:** the trace looks clean — no `error`, natural termination — but the answer is wrong. The bug is in the `inputs`. Here the task asked about **Tokyo**, but the `lookup` span's `inputs` say `{'city': 'Paris'}`. The call succeeded and returned a valid population — *for the wrong city*.

This is the hardest signature to spot and the whole argument for tracing **arguments**, not just call names or a success flag. A success flag would have called this run green.

In [7]:
narrate(wrong_args)

# No error field is set -- the failure is only visible by reading the arguments.
lookup_span = next(span for span in wrong_args.trace if span.name == "lookup")
print("\nerror field:", lookup_span.error, "(clean!)")
print("but inputs say:", lookup_span.inputs, "-- the task asked about Tokyo, not Paris")

[0] chain agent_graph.run      {'termination': 'natural', 'iterations': 2}
[1] llm   agent                -> tool_calls=['lookup'] tokens=120
[2] tool  lookup               {'city': 'Paris'} -> '11000000'
[3] llm   agent                -> tool_calls=[] tokens=120

error field: None (clean!)
but inputs say: {'city': 'Paris'} -- the task asked about Tokyo, not Paris


### 4.3 Runaway loop — the same `(name, inputs)` repeats, ending in `max_steps`

**Signature:** the same `(name, inputs)` pair repeats across iterations, and the run ends in `termination == "max_steps"` instead of `"natural"`. The model keeps asking for a call that never makes progress. The repetition is obvious at a glance in the trace.

In [8]:
narrate(runaway)

# Same (name, inputs) over and over, and it never reached a natural stop.
tool_calls = [
    (span.name, tuple(span.inputs.items())) for span in runaway.trace if span.run_type == "tool"
]
print("\ndistinct tool calls:", len(set(tool_calls)), "across", len(tool_calls), "dispatches")
print("termination:", runaway.termination)

[0] chain agent_graph.run      {'termination': 'max_steps', 'iterations': 4}
[1] llm   agent                -> tool_calls=['lookup'] tokens=120
[2] tool  lookup               {'city': 'Atlantis'} -> 'Error: KeyError("no population on file for \'Atlantis\'")\n Please fix your mistakes.'  [is_error]
[3] llm   agent                -> tool_calls=['lookup'] tokens=120
[4] tool  lookup               {'city': 'Atlantis'} -> 'Error: KeyError("no population on file for \'Atlantis\'")\n Please fix your mistakes.'  [is_error]
[5] llm   agent                -> tool_calls=['lookup'] tokens=120
[6] tool  lookup               {'city': 'Atlantis'} -> 'Error: KeyError("no population on file for \'Atlantis\'")\n Please fix your mistakes.'  [is_error]
[7] llm   agent                -> tool_calls=['lookup'] tokens=120
[8] tool  lookup               {'city': 'Atlantis'} -> 'Error: KeyError("no population on file for \'Atlantis\'")\n Please fix your mistakes.'  [is_error]

distinct tool calls: 1 across 4 di

### 4.4 Premature termination — `natural` stop, but no tool call

**Signature:** the run terminates `natural` (the model thinks it's done) — but it never called the tool it needed, so the answer is a guess. The trace has *zero* `tool` spans: the model went straight from the question to a text answer. "Stopped cleanly" is not the same as "stopped correctly."

In [9]:
narrate(premature)

n_tool_spans = sum(1 for span in premature.trace if span.run_type == "tool")
print("\ntermination:", premature.termination, "| tool spans:", n_tool_spans)
print("answer:", premature.final_text, "<- a guess; it never looked anything up")

[0] chain agent_graph.run      {'termination': 'natural', 'iterations': 1}
[1] llm   agent                -> tool_calls=[] tokens=120

termination: natural | tool spans: 0
answer: It's probably around 5 million. <- a guess; it never looked anything up


[↑ Back to top](#top)

## 5. Takeaways

- **A trace is the durable record of an ephemeral run.** The graph runs and exits; the trace is what lets you answer "what did it *do*?" later — by reading, not re-running.
- **A trace is the agent's state, captured.** The growing message history the graph feeds the model each turn is the live **state**; the trace serializes it so you can read what the model saw after the fact. Reading the trace *is* reading the state.
- **Read top to bottom with `.one_line()`.** `trace[0]` is the `chain` summary; then `llm` and `tool` spans alternate in run order. Narrate them and you've reconstructed the run.
- **Arguments are where the truth is.** A `tool` span's `inputs` are the model's actual choices. Read them first — they catch the bug a call-count or a success flag misses.
- **The four failure signatures**, each readable from the trace alone:
  - **tool error** — a `tool` span's `error` field is set.
  - **wrong arguments** — clean run, but the `inputs` answer the wrong question.
  - **runaway loop** — the same `(name, inputs)` repeats, ending in `max_steps`.
  - **premature termination** — `natural` stop with no (or too few) tool calls; the answer is a guess.

**Next:**

- **`L1203_lab`** — you practice exactly this: narrate the good run, detect the runaway, catch the wrong argument, classify all four signatures.
- **`L1204_lecture`** — instrument the graph yourself (a wrapper, not a rewrite) and compare two traces of the same task: signal vs. noise.

These four signatures aren't just debugging lore — in **L13** they become your **first eval cases**. The failure you can name in a trace today is the test you write tomorrow.

[↑ Back to top](#top)